# Library｜只修四句，各三候選

只處理Rosie「A crown. Crown.」與Princess三句，共12個候選。兩句已接受Rosie原音鎖定SHA256，不重錄、不覆蓋。其他Library24段、Park與Bakery均不生成。

1. 新執行階段先選免費T4 GPU並安裝；若仍是原Library成功環境可跳過安裝。
2. 上傳 **library-retakes4-inputs.zip**，已含你這次六句原音/驗收紀錄與原核准Rosie參考，不必重新選角或再上傳其他原音包。此為私人包，請勿公開。
3. 檢查dry-run四句，再將RUN_RETAKES=True，生成獨立12候選。
4. 比較原音、候選1/2/3，四句各選一版後才匯出新六句副本。未選不覆蓋，皆不喜歡可保留原音並再試。

Rosie用原Qwen參考與不同seed，無pitch/speed處理。Princess固定af_bella：候選1原文整句speed0.92；候選2 speed0.86、原句尾增加0.28秒停頓；候選3 speed0.82、原句尾0.45秒/逗點0.18秒停頓。這是重新生成時的speed與空白停頓，沒有改pitch，也不把變慢當作已達成溫柔。請逐句判斷自然感、溫柔感、字尾及節奏。

## 0｜安裝（同一已成功的Library環境可跳過）

In [ ]:
import sys, subprocess, os
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
subprocess.check_call(['apt-get','-qq','update'])
subprocess.check_call(['apt-get','-qq','install','-y','espeak-ng','sox','libsndfile1'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','qwen-tts==0.1.1','transformers==4.57.3','accelerate==1.12.0','kokoro==0.9.4','soundfile','ipywidgets>=8,<9','huggingface-hub>=0.34,<1'])
print('套件安裝完成。若Colab提示套件已載入，請重新啟動執行階段，再從上傳格繼續。')

## 1｜上傳修正輸入包並驗證原音

In [ ]:
from pathlib import Path
from google.colab import files
import io,zipfile,uuid,json,sys,importlib.util,shutil
WORK=Path('/content')/('library_retakes4_inputs_'+uuid.uuid4().hex[:8])
uploaded=files.upload()
if len(uploaded)!=1:raise ValueError('請只選 library-retakes4-inputs.zip')
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as z:
    entries=[i for i in z.infolist() if not i.is_dir()]
    if sum(i.file_size for i in entries)>50_000_000 or len({i.filename for i in entries})!=len(entries):raise ValueError('輸入ZIP無效')
    paths=[(i,(WORK/i.filename).resolve()) for i in entries]
    if any(not p.is_relative_to(WORK.resolve()) for _,p in paths):raise ValueError('不安全的ZIP路徑')
    for i,p in paths:
        p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(z.read(i))
sys.path.insert(0,str(WORK));sys.modules.pop('library_runner',None)
spec=importlib.util.spec_from_file_location('library_retakes_four',WORK/'retake_runner.py')
repair=importlib.util.module_from_spec(spec);spec.loader.exec_module(repair)
manifest,jobs,original=repair.verify(WORK)
print('原六句WAV/英文/ID/事件/人工驗收與主參考已核對。')
for line_id,hash_value in repair.ACCEPTED.items():print('保留原檔：',line_id,hash_value)

## 2｜四句dry-run：只會產生12候選，不觸碰已接受兩句

In [ ]:
planned=repair.generate(WORK,'/content/library_retakes4_candidates',run=False)
for job in planned['jobs']:print(job['lineId'],job['voiceId'],repr(job['text']),'／3候選')
print('待修',planned['lineCount'],'句；候選總數',planned['candidateCount'],'；目前生成',planned['generated'])

## 3｜生成12候選（把開關改True才推論）

In [ ]:
RUN_RETAKES=False
if RUN_RETAKES:
    LIBRARY_RETAKE_BATCH=repair.generate(WORK,'/content/library_retakes4_candidates',run=True)
    print('候選已另存：',LIBRARY_RETAKE_BATCH,'。原六句未覆蓋，兩接受音檔不變。')
else:
    print('目前未生成。要製作12候選時，改RUN_RETAKES=True再執行本格。')

## 4｜比較原音/候選1、2、3，選定後才匯出

In [ ]:
from IPython.display import Audio,display
import ipywidgets as widgets
if 'LIBRARY_RETAKE_BATCH' not in globals() or not (Path(LIBRARY_RETAKE_BATCH)/'candidate-manifest.json').exists():
    print('尚未生成候選。請先執行區塊3；本格不會自行生成。')
else:
    candidates=repair.base.read(Path(LIBRARY_RETAKE_BATCH)/'candidate-manifest.json')
    if candidates.get('scope')!='library-retakes4-candidates':raise ValueError('不是本次Library四句候選資料夾')
    print('先聽核准Rosie參考音與兩句已接受音檔（只比較、不變動）：')
    ref=manifest['characters']['rosie']['reference']
    display(Audio(filename=str(WORK/'references'/ref['file'])))
    for line_id in repair.ACCEPTED:
        a=next(a for a in original['assets'].values() if a['lineId']==line_id)
        print('已接受：',a['text']);display(Audio(filename=str(WORK/'originals'/a['src'])))
    selectors={}
    for line_id in repair.TARGETS:
        a=next(a for a in original['assets'].values() if a['lineId']==line_id)
        print('\n',line_id,'\n',a['voiceId'],a['text'],'\n原音：')
        display(Audio(filename=str(WORK/'originals'/a['src'])))
        versions=candidates['candidates'].get(line_id,[])
        for c in versions:
            print('候選',c['candidate'],'／',c['retakeSettings'])
            display(Audio(filename=str(Path(LIBRARY_RETAKE_BATCH)/c['filename'])))
        selectors[line_id]=widgets.Dropdown(options=[('尚未選擇',None)]+[('候選 '+str(c['candidate']),c['candidate']) for c in versions],description='採用：')
        display(selectors[line_id])
    out=widgets.Output();export_button=widgets.Button(description='確認四句並匯出新六句副本',layout=widgets.Layout(width='260px'));save_button=widgets.Button(description='只下載候選、不採用',layout=widgets.Layout(width='220px'))
    def download_candidates(_):
        with out:files.download(shutil.make_archive(str(LIBRARY_RETAKE_BATCH),'zip',LIBRARY_RETAKE_BATCH))
    def export_choices(_):
        with out:
            out.clear_output()
            choices={k:s.value for k,s in selectors.items()}
            if any(v is None for v in choices.values()):
                print('四句都選定後才能匯出；未選不覆蓋。若不喜歡，可只下載候選或重跑區塊3。');return
            try:
                dest=Path('/content')/('library_selected6_'+uuid.uuid4().hex[:8])
                repair.export_selected(WORK,LIBRARY_RETAKE_BATCH,choices,dest)
                print('新副本：',dest,'；兩句已接受WAV/hash不變，原始六句未覆蓋。')
                files.download(shutil.make_archive(str(dest),'zip',dest))
            except Exception as exc:print('未匯出：',exc)
    save_button.on_click(download_candidates);export_button.on_click(export_choices)
    display(save_button,export_button,out)

## 驗收與限制

本次僅交付流程，沒有在Colab GPU實際產出或試聽12候選；已做語法、原ZIP/hash、範圍、停頓文字保真及假模型匯出測試。Kokoro不能直接保證溫柔情緒，慢速與停頓候選仍需人工挑選。若不自然請勿勉強採用，也不自動換聲線。

新副本維持原ID、assetId、字幕、事件與聲線；四句更新長度/hash並清除舊visemes，需未來遊戲內驗收，不宣稱精準音素對齊。本流程不寫入正式遊戲。